In [1]:
import os
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import gc
import json
import math
import random
import re
from collections import Counter, defaultdict
from dataclasses import dataclass, asdict
from difflib import SequenceMatcher
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
from datasets import load_dataset
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# Optional richer interactive figures.
try:
    import plotly.graph_objects as go
    PLOTLY_AVAILABLE = True
except Exception:
    PLOTLY_AVAILABLE = False

try:
    from peft import PeftModel
    HAS_PEFT = True
except Exception:
    HAS_PEFT = False

# ============================================================
# CONFIG
# ============================================================

BASE_MODEL = "microsoft/phi-3-mini-4k-instruct"
SFT_PATH = "/kaggle/input/datasets/adithyaled24b039/phi3-sft-folderr/phi3_sft_lora"

OUT_DIR = "/kaggle/working/strategyqa_self_correction_basin_atlas"
CSV_DIR = os.path.join(OUT_DIR, "csv")
PLOTS_DIR = os.path.join(OUT_DIR, "plots")
REPORTS_DIR = os.path.join(OUT_DIR, "reports")
HTML_DIR = os.path.join(OUT_DIR, "html")
CACHE_DIR = os.path.join(OUT_DIR, "cache")
NPZ_DIR = os.path.join(OUT_DIR, "npz")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if torch.cuda.is_available() else torch.float32

SEED = 42
EVAL_SEED = 42

# Sample sizes (tuned for richer plots without being unreasonable on T4-class hardware).
N_VALIDATION = 120
N_TEST = 120

# Prompt variants define a scaffold-strength axis.
@dataclass(frozen=True)
class PromptVariant:
    name: str
    prefix: str
    style: str
    spacing: str
    reasoning: bool
    self_check: bool
    strength: float
    note: str

VARIANTS = [
    PromptVariant("plain", "", "plain", "compact", False, False, 0.00, "minimal prompt"),
    PromptVariant("anchored", "##", "plain", "compact", False, False, 0.15, "anchored header"),
    PromptVariant("anchored_reason", "##", "plain", "roomy", True, False, 0.35, "anchored + reasoning"),
    PromptVariant("anchored_xml", "##", "xml", "roomy", True, False, 0.55, "anchored + XML answer"),
    PromptVariant("anchored_selfcheck", "##", "xml", "roomy", True, True, 0.75, "anchored + self-check"),
    PromptVariant("answer_first", "", "answer_first", "compact", True, False, 0.45, "answer directive first"),
    PromptVariant("verbose", "###", "xml", "spacious", True, True, 1.00, "verbose scaffold"),
]
BASELINE_VARIANT = "plain"

# Decoding.
PASS1_MAX_NEW = 48
PASS2_MAX_NEW = 48
TEMPERATURE = 0.0
TOP_P = 0.9

# Calibration.
N_THRESHOLD_STEPS = 201
N_RELIABILITY_BINS = 10

# Plot / atlas selection.
REPRESENTATIVE_QUESTIONS = 12

# Optional quiver density.
QUIVER_STEP = 2

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

for p in [OUT_DIR, CSV_DIR, PLOTS_DIR, REPORTS_DIR, HTML_DIR, CACHE_DIR, NPZ_DIR]:
    os.makedirs(p, exist_ok=True)

TOKENIZER = None
MODEL = None
BLOCKS = None
FINAL_NORM = None
LM_HEAD = None
MODEL_LAYERS = None
MODEL_HIDDEN = None

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({
    "figure.dpi": 140,
    "savefig.dpi": 180,
    "axes.titlesize": 13,
    "axes.labelsize": 11,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 9,
    "font.family": "DejaVu Sans",
})

# ============================================================
# UTILS
# ============================================================

def ensure_dir(path):
    os.makedirs(path, exist_ok=True)


def save_df(df, path):
    ensure_dir(os.path.dirname(path))
    df.to_csv(path, index=False)


def save_json(obj, path):
    ensure_dir(os.path.dirname(path))
    with open(path, "w") as f:
        json.dump(obj, f, indent=2)


def preview_text(s, max_len=180):
    s = str(s).replace("\n", " ")
    return s[:max_len] + ("..." if len(s) > max_len else "")


def to_jsonable(x):
    try:
        return json.dumps(x, ensure_ascii=False)
    except Exception:
        return json.dumps(str(x), ensure_ascii=False)


def free_memory(*objs):
    for o in objs:
        try:
            del o
        except Exception:
            pass
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        try:
            torch.cuda.synchronize()
        except Exception:
            pass


def normalize_text(x):
    return re.sub(r"\s+", " ", str(x).strip().lower())


def safe_mean(values):
    s = pd.Series(values)
    return float(s.mean()) if len(s) else float("nan")


def safe_std(values):
    s = pd.Series(values)
    return float(s.std()) if len(s) else float("nan")


def safe_pearson(x, y):
    x = np.asarray(x, dtype=np.float64)
    y = np.asarray(y, dtype=np.float64)
    mask = np.isfinite(x) & np.isfinite(y)
    x = x[mask]
    y = y[mask]
    if len(x) < 2 or np.std(x) < 1e-12 or np.std(y) < 1e-12:
        return float("nan")
    return float(np.corrcoef(x, y)[0, 1])


def entropy_from_probs(p):
    p = np.asarray(p, dtype=np.float64)
    p = p[p > 0]
    if len(p) == 0:
        return float("nan")
    return float(-np.sum(p * np.log2(p + 1e-12)))


def softmax_np(x):
    x = np.asarray(x, dtype=np.float64)
    x = x - np.max(x)
    ex = np.exp(x)
    return ex / (ex.sum() + 1e-12)


def annotated_bar(ax, fmt="{:.3f}"):
    for p in ax.patches:
        h = p.get_height()
        if np.isfinite(h):
            ax.annotate(fmt.format(h), (p.get_x() + p.get_width() / 2.0, h),
                        ha="center", va="bottom", fontsize=8, xytext=(0, 3), textcoords="offset points")


def bar_plot(labels, values, title, path, ylabel="Value", rotate=30, color_map="Set2"):
    plt.figure(figsize=(10.5, 4.8))
    ax = plt.gca()
    ax.bar(labels, values, color=plt.get_cmap(color_map)(np.linspace(0.12, 0.88, len(labels))))
    ax.set_title(title)
    ax.set_ylabel(ylabel)
    ax.tick_params(axis="x", rotation=rotate)
    annotated_bar(ax)
    plt.tight_layout()
    plt.savefig(path, dpi=180)
    plt.close()


def grouped_bar_plot(labels, v1, v2, label1, label2, title, path, ylabel="Accuracy"):
    x = np.arange(len(labels))
    width = 0.35
    fig, ax = plt.subplots(figsize=(11, 5.5))
    ax.bar(x - width/2, v1, width, label=label1, color="tab:blue", alpha=0.85)
    ax.bar(x + width/2, v2, width, label=label2, color="tab:orange", alpha=0.85)
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=30, ha="right")
    ax.legend(frameon=True)
    annotated_bar(ax)
    plt.tight_layout()
    plt.savefig(path, dpi=180)
    plt.close()


def violin_plot_distributions(df, metric, group_col, title, path, ylabel="Value"):
    plt.figure(figsize=(11, 6))
    groups = df[group_col].unique()
    data = [df[df[group_col] == g][metric].dropna().to_numpy(dtype=np.float64) for g in groups]
    
    valid_idx = [i for i, d in enumerate(data) if len(d) > 0]
    valid_groups = [groups[i] for i in valid_idx]
    valid_data = [data[i] for i in valid_idx]

    if valid_data:
        parts = plt.violinplot(valid_data, showmeans=True, showmedians=False)
        for pc in parts['bodies']:
            pc.set_facecolor('tab:blue')
            pc.set_edgecolor('black')
            pc.set_alpha(0.6)
        plt.xticks(range(1, len(valid_groups) + 1), valid_groups, rotation=30, ha="right")

    plt.title(title)
    plt.ylabel(ylabel)
    plt.tight_layout()
    plt.savefig(path, dpi=180)
    plt.close()


def line_plot(x, ys, labels, title, path, xlabel="X", ylabel="Y"):
    plt.figure(figsize=(10.5, 5.2))
    for y, label in zip(ys, labels):
        plt.plot(x, y, marker="o", linewidth=1.8, label=label)
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.legend(frameon=True)
    plt.tight_layout()
    plt.savefig(path, dpi=180)
    plt.close()


def histogram(values, title, path, xlabel="Value", bins=12):
    plt.figure(figsize=(8.8, 4.4))
    plt.hist(values, bins=bins, alpha=0.88)
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel("Count")
    plt.tight_layout()
    plt.savefig(path, dpi=180)
    plt.close()


def scatter(x, y, title, path, xlabel="X", ylabel="Y", color=None, alpha=0.8):
    plt.figure(figsize=(7.2, 5.6))
    plt.scatter(x, y, alpha=alpha, c=color, s=26)
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.tight_layout()
    plt.savefig(path, dpi=180)
    plt.close()


def heatmap(mat, xlabels, ylabels, title, path, xlabel="", ylabel="", cmap="viridis", fmt=None):
    plt.figure(figsize=(max(8.5, len(xlabels) * 0.55), max(5.2, len(ylabels) * 0.38)))
    im = plt.imshow(mat, aspect="auto", cmap=cmap)
    plt.colorbar(im)
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.xticks(range(len(xlabels)), xlabels, rotation=45, ha="right")
    plt.yticks(range(len(ylabels)), ylabels)
    if fmt is not None:
        for i in range(mat.shape[0]):
            for j in range(mat.shape[1]):
                if np.isfinite(mat[i, j]):
                    plt.text(j, i, fmt.format(mat[i, j]), ha="center", va="center", fontsize=7,
                             color="white" if abs(mat[i, j]) > 0.5 else "black")
    plt.tight_layout()
    plt.savefig(path, dpi=180)
    plt.close()


def polyfit_line(x, y, deg=2):
    x = np.asarray(x, dtype=np.float64)
    y = np.asarray(y, dtype=np.float64)
    mask = np.isfinite(x) & np.isfinite(y)
    x = x[mask]
    y = y[mask]
    if len(x) < 3:
        return None, None
    deg = min(deg, len(x) - 1)
    coeffs = np.polyfit(x, y, deg=deg)
    xs = np.linspace(x.min(), x.max(), 200)
    ys = np.polyval(coeffs, xs)
    return xs, ys


def sequence_similarity(a, b):
    return float(SequenceMatcher(None, normalize_text(a), normalize_text(b)).ratio())


def jaccard_token_similarity(a, b):
    sa = set(re.findall(r"\w+", normalize_text(a)))
    sb = set(re.findall(r"\w+", normalize_text(b)))
    if not sa and not sb:
        return 1.0
    if not sa or not sb:
        return 0.0
    return float(len(sa & sb) / len(sa | sb))


def ridge_from_distributions(distributions, labels, title, path):
    plt.figure(figsize=(10.8, 7.4))
    xs = np.linspace(0, 1, 400)
    cmap = plt.get_cmap("viridis")
    for i, (dist, label) in enumerate(zip(distributions, labels)):
        dist = np.asarray(dist, dtype=np.float64)
        hist, bins = np.histogram(dist, bins=60, range=(0, 1), density=True)
        centers = (bins[:-1] + bins[1:]) / 2.0
        curve = np.interp(xs, centers, hist)
        kernel = np.exp(-0.5 * ((np.arange(-10, 11)) / 3.0) ** 2)
        kernel /= kernel.sum()
        curve = np.convolve(curve, kernel, mode="same")
        y = (curve / (curve.max() + 1e-12)) * 0.75
        offset = i * 1.12
        plt.fill_between(xs, offset, offset + y, color=cmap(i / max(1, len(distributions) - 1)), alpha=0.88)
        plt.plot(xs, offset + y, color="white", linewidth=1.0)
        plt.text(1.01, offset + 0.18, label, fontsize=9, va="center")
    plt.title(title)
    plt.yticks([])
    plt.xlabel("Selected answer proportion")
    plt.xlim(0, 1)
    plt.tight_layout()
    plt.savefig(path, dpi=180)
    plt.close()


def quiver_transition_plot(df, path, color_by="transition"):
    sub = df.copy()
    fig, ax = plt.subplots(figsize=(8.8, 6.4))
    color_map = {
        "correct→correct": "tab:green",
        "wrong→correct": "tab:blue",
        "correct→wrong": "tab:red",
        "wrong→wrong": "tab:gray",
    }
    colors = [color_map.get(t, "tab:gray") for t in sub["transition"].tolist()]
    ax.scatter(sub["pass1_logodds"], sub["pass1_commitment"], c=colors, alpha=0.7, s=18)
    dx = (sub["pass2_logodds"] - sub["pass1_logodds"]).to_numpy(dtype=np.float64)
    dy = (sub["pass2_commitment"] - sub["pass1_commitment"]).to_numpy(dtype=np.float64)
    ax.quiver(sub["pass1_logodds"].to_numpy(dtype=np.float64), sub["pass1_commitment"].to_numpy(dtype=np.float64), dx, dy,
              angles="xy", scale_units="xy", scale=1.0, alpha=0.35, width=0.002)
    ax.axvline(0, color="black", linewidth=1, alpha=0.35)
    ax.axhline(0.5, color="black", linewidth=1, alpha=0.2)
    ax.set_title("Self-correction trajectory field")
    ax.set_xlabel("Pass-1 log-odds")
    ax.set_ylabel("Pass-1 commitment")
    plt.tight_layout()
    plt.savefig(path, dpi=180)
    plt.close()


def strip_transition_plot(df, path, max_points=250):
    sub = df.sample(min(max_points, len(df)), random_state=SEED).copy() if len(df) > max_points else df.copy()
    fig, ax = plt.subplots(figsize=(12.5, 6.8))
    xs1 = np.zeros(len(sub))
    xs2 = np.ones(len(sub))
    for i, (_, r) in enumerate(sub.iterrows()):
        color = {
            "correct→correct": "tab:green",
            "wrong→correct": "tab:blue",
            "correct→wrong": "tab:red",
            "wrong→wrong": "tab:gray",
        }.get(r["transition"], "tab:gray")
        ax.plot([0, 1], [r["pass1_logodds"], r["pass2_logodds"]], color=color, alpha=0.18, linewidth=1.0)
        ax.scatter([0, 1], [r["pass1_logodds"], r["pass2_logodds"]], color=color, alpha=0.65, s=12)
    ax.axhline(0, color="black", linewidth=1, alpha=0.35)
    ax.set_xticks([0, 1])
    ax.set_xticklabels(["Pass 1", "Pass 2"])
    ax.set_ylabel("Log-odds")
    ax.set_title("Pass-wise log-odds strip plot")
    plt.tight_layout()
    plt.savefig(path, dpi=180)
    plt.close()


def waterfall_plot(summary_df, path):
    sdf = summary_df.sort_values("strength")
    delta = sdf["pass2_accuracy"] - sdf["pass1_accuracy"]
    colors = ["tab:green" if v >= 0 else "tab:red" for v in delta]
    plt.figure(figsize=(10.8, 5.0))
    plt.bar(sdf["variant"], delta, color=colors)
    plt.axhline(0, color="black", linewidth=1, alpha=0.5)
    plt.ylabel("Accuracy change (pass2 - pass1)")
    plt.title("Self-correction waterfall")
    plt.xticks(rotation=30, ha="right")
    annotated_bar(plt.gca(), fmt="{:.3f}")
    plt.tight_layout()
    plt.savefig(path, dpi=180)
    plt.close()


def response_drift_scatter(df, path):
    sub = df.copy()
    drift = sub["response_drift_sequence"].to_numpy(dtype=np.float64)
    delta = (sub["pass2_logodds"] - sub["pass1_logodds"]).to_numpy(dtype=np.float64)
    colors = sub["transition"].map({
        "correct→correct": 0,
        "wrong→correct": 1,
        "correct→wrong": 2,
        "wrong→wrong": 3,
    }).fillna(3).to_numpy(dtype=np.int64)
    plt.figure(figsize=(8.2, 6.0))
    sc = plt.scatter(drift, delta, c=colors, cmap="tab10", alpha=0.75, s=25)
    plt.axhline(0, color="black", linewidth=1, alpha=0.35)
    plt.xlabel("Explanation drift (SequenceMatcher distance)")
    plt.ylabel("Δ log-odds (pass2 - pass1)")
    plt.title("Response drift vs confidence shift")
    cbar = plt.colorbar(sc)
    cbar.set_ticks([0, 1, 2, 3])
    cbar.set_ticklabels(["cc", "wc", "cw", "ww"])
    plt.tight_layout()
    plt.savefig(path, dpi=180)
    plt.close()


def state_basin_heatmap(df, path, xcol, ycol, zcol, title, xbins=8, ybins=8):
    sub = df.copy()
    x = sub[xcol].to_numpy(dtype=np.float64)
    y = sub[ycol].to_numpy(dtype=np.float64)
    z = sub[zcol].to_numpy(dtype=np.float64)
    x_edges = np.quantile(x, np.linspace(0, 1, xbins + 1))
    y_edges = np.quantile(y, np.linspace(0, 1, ybins + 1))
    x_edges = np.unique(x_edges)
    y_edges = np.unique(y_edges)
    if len(x_edges) < 3 or len(y_edges) < 3:
        return None
    xi = np.clip(np.digitize(x, x_edges) - 1, 0, len(x_edges) - 2)
    yi = np.clip(np.digitize(y, y_edges) - 1, 0, len(y_edges) - 2)
    mat = np.full((len(y_edges) - 1, len(x_edges) - 1), np.nan, dtype=np.float64)
    for i in range(len(y_edges) - 1):
        for j in range(len(x_edges) - 1):
            mask = (xi == j) & (yi == i)
            if mask.sum() > 0:
                mat[i, j] = float(z[mask].mean())
    plt.figure(figsize=(9.0, 6.8))
    im = plt.imshow(mat, aspect="auto", origin="lower", cmap="coolwarm", norm=TwoSlopeNorm(vmin=np.nanmin(mat), vcenter=0.5, vmax=np.nanmax(mat)))
    plt.colorbar(im)
    plt.title(title)
    plt.xlabel(xcol)
    plt.ylabel(ycol)
    plt.tight_layout()
    plt.savefig(path, dpi=180)
    plt.close()
    return mat

# ============================================================
# CACHE / SAMPLING
# ============================================================

def cached_indices(name, ds_len, n, seed=EVAL_SEED):
    ensure_dir(CACHE_DIR)
    n = min(n, ds_len)
    path = os.path.join(CACHE_DIR, f"{name}_n{n}_seed{seed}.json")
    if os.path.exists(path):
        with open(path, "r") as f:
            return json.load(f)
    rng = np.random.RandomState(seed)
    idx = rng.choice(ds_len, size=n, replace=False).tolist()
    with open(path, "w") as f:
        json.dump(idx, f)
    return idx


def sample_dataset(ds, name, n, seed=EVAL_SEED):
    idx = cached_indices(name, len(ds), n, seed=seed)
    return ds.select(idx), idx

# ============================================================
# ANSWER HELPERS
# ============================================================

def extract_yes_no(text):
    if text is None:
        return None
    m = re.findall(r"<answer>(.*?)</answer>", text, flags=re.IGNORECASE | re.DOTALL)
    span = m[-1] if m else text
    hits = re.findall(r"\b(yes|no)\b", span, flags=re.IGNORECASE)
    if hits:
        return hits[-1].lower()
    hits = re.findall(r"\b(yes|no)\b", text, flags=re.IGNORECASE)
    return hits[-1].lower() if hits else None


def yesno_to_int(x):
    return 1 if str(x).lower().strip() == "yes" else 0


def token_ids_for_text(tokenizer, text):
    forms = [
        text, f" {text}", f"\n{text}", text.upper(), f" {text.upper()}", text.lower(), f" {text.lower()}"
    ]
    ids = []
    for form in forms:
        enc = tokenizer.encode(form, add_special_tokens=False)
        if len(enc) == 1:
            ids.append(enc[0])
    if not ids:
        enc = tokenizer.encode(text, add_special_tokens=False)
        if len(enc) > 0:
            ids.append(enc[0])
    return sorted(set(ids))


def target_from_ids(logits, ids):
    if len(ids) == 0:
        return None, float("nan")
    vals = [float(logits[i].item()) for i in ids]
    idx = int(np.argmax(vals))
    return ids[idx], vals[idx]


def group_logsumexp(logits, ids):
    if len(ids) == 0:
        return float("-inf")
    vals = torch.tensor([float(logits[i].item()) for i in ids], device=logits.device)
    return float(torch.logsumexp(vals, dim=0).item())


def candidate_group_metrics(logits, candidate_groups, target_group_idx):
    scores = [group_logsumexp(logits, ids) for ids in candidate_groups]
    probs = softmax_np(scores)
    return {
        "scores": scores,
        "probs": probs.tolist(),
        "entropy": entropy_from_probs(probs),
        "margin": float(scores[target_group_idx] - max([s for i, s in enumerate(scores) if i != target_group_idx])),
        "target_prob": float(probs[target_group_idx]),
        "flip_uncertainty": float(1.0 - abs(probs[0] - probs[1])),
    }


def commitment_score(text):
    t = normalize_text(text)
    score = 0.5
    if "<answer>" in t and "</answer>" in t:
        score += 0.15
    if "<think>" in t:
        score += 0.10
    if re.search(r"\b(yes|no)\b", t):
        score += 0.15
    if any(h in t for h in ["maybe", "probably", "perhaps", "not sure", "uncertain", "depends"]):
        score -= 0.20
    if len(t.split()) <= 6:
        score += 0.10
    if len(t.split()) > 30:
        score -= 0.10
    return max(0.0, min(1.0, score))


def answer_hedged(text):
    t = normalize_text(text)
    return bool(any(h in t for h in ["maybe", "probably", "perhaps", "not sure", "uncertain", "depends"]))


def explicit_answer_emitted(text):
    return bool(re.search(r"\b(yes|no)\b", normalize_text(text)))

# ============================================================
# MODEL LOADING
# ============================================================

def load_tokenizer():
    global TOKENIZER
    if TOKENIZER is None:
        src = SFT_PATH if (HAS_PEFT and os.path.exists(SFT_PATH)) else BASE_MODEL
        TOKENIZER = AutoTokenizer.from_pretrained(src)
        if TOKENIZER.pad_token is None:
            TOKENIZER.pad_token = TOKENIZER.eos_token
    return TOKENIZER


def discover_blocks(model):
    candidates = [
        ("model.layers", getattr(getattr(model, "model", None), "layers", None)),
        ("model.decoder.layers", getattr(getattr(getattr(model, "model", None), "decoder", None), "layers", None)),
        ("transformer.h", getattr(getattr(model, "transformer", None), "h", None)),
        ("gpt_neox.layers", getattr(getattr(model, "gpt_neox", None), "layers", None)),
        ("decoder.layers", getattr(getattr(model, "decoder", None), "layers", None)),
    ]
    for name, maybe_blocks in candidates:
        if maybe_blocks is not None:
            maybe_blocks = list(maybe_blocks)
            if len(maybe_blocks) > 0:
                return maybe_blocks, name
    raise RuntimeError("Could not locate transformer blocks.")


def discover_final_norm(model):
    candidates = [
        ("model.norm", getattr(getattr(model, "model", None), "norm", None)),
        ("transformer.ln_f", getattr(getattr(model, "transformer", None), "ln_f", None)),
        ("gpt_neox.final_layer_norm", getattr(getattr(model, "gpt_neox", None), "final_layer_norm", None)),
        ("decoder.final_layer_norm", getattr(getattr(model, "decoder", None), "final_layer_norm", None)),
    ]
    for name, mod in candidates:
        if mod is not None:
            return mod, name
    return None, None


def discover_lm_head(model):
    if hasattr(model, "lm_head"):
        return model.lm_head, "lm_head"
    if hasattr(model, "embed_out"):
        return model.embed_out, "embed_out"
    raise RuntimeError("Could not locate lm_head / embed_out.")


def load_model():
    global MODEL, BLOCKS, FINAL_NORM, LM_HEAD, MODEL_LAYERS, MODEL_HIDDEN
    load_tokenizer()

    kwargs = {"torch_dtype": DTYPE}
    try:
        kwargs["attn_implementation"] = "eager"
        model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, **kwargs)
    except Exception:
        kwargs.pop("attn_implementation", None)
        model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, **kwargs)

    model = model.to(DEVICE)
    model.eval()

    if HAS_PEFT and os.path.exists(SFT_PATH):
        try:
            model = PeftModel.from_pretrained(model, SFT_PATH).merge_and_unload().to(DEVICE)
            model.eval()
        except Exception:
            pass

    try:
        model.config.use_cache = False
    except Exception:
        pass

    blocks, block_name = discover_blocks(model)
    final_norm, norm_name = discover_final_norm(model)
    lm_head, head_name = discover_lm_head(model)

    MODEL = model
    BLOCKS = blocks
    FINAL_NORM = final_norm
    LM_HEAD = lm_head
    MODEL_LAYERS = len(BLOCKS)
    try:
        MODEL_HIDDEN = int(model.config.hidden_size)
    except Exception:
        MODEL_HIDDEN = int(lm_head.weight.shape[1])

    print(f"Loaded model with {MODEL_LAYERS} blocks ({block_name}), norm={norm_name}, head={head_name}")
    return model, TOKENIZER

# ============================================================
# DATA
# ============================================================

def load_strategyqa_split(split_name, n, name_tag):
    ds = load_dataset("ChilleD/StrategyQA", split=split_name)
    ds, idx = sample_dataset(ds, name_tag, n)
    samples = []
    for i, s in enumerate(ds):
        samples.append({
            "question_id": i,
            "source_index": idx[i],
            "question": s["question"],
            "gold": "yes" if bool(s["answer"]) else "no",
        })
    return samples

# ============================================================
# PROMPTS
# ============================================================

def task_instruction(variant: PromptVariant):
    if variant.self_check:
        return "Reason carefully, then check your decision before answering."
    if variant.reasoning:
        return "Reason carefully before answering."
    return "Decide the answer."


def answer_directive(variant: PromptVariant):
    if variant.style == "xml":
        return "Put ONLY yes or no inside <answer></answer>."
    if variant.style == "answer_first":
        return "Answer:"
    return "Answer only yes or no."


def build_pass1_prompt(question: str, variant: PromptVariant):
    header = f"{variant.prefix} Question:" if variant.prefix else "Question:"
    blocks = [
        f"{header} {question}",
        task_instruction(variant),
        answer_directive(variant),
    ]
    if variant.reasoning:
        blocks.insert(2, "Think briefly before you answer.")
    sep = "\n\n" if variant.spacing in ("roomy", "spacious") else "\n"
    return sep.join([b for b in blocks if b and str(b).strip()])


def build_pass2_prompt(question: str, variant: PromptVariant, pass1_completion: str, pass1_pred: Optional[str]):
    header = f"{variant.prefix} Question:" if variant.prefix else "Question:"
    blocks = [
        f"{header} {question}",
        f"Previous attempt:\n{pass1_completion}",
        "Review the previous attempt and identify any mistake.",
        "Then give the corrected answer.",
        answer_directive(variant),
    ]
    if variant.self_check:
        blocks.insert(3, "Verify your answer before finalizing it.")
    if pass1_pred is not None:
        blocks.insert(2, f"Previous answer guess: {pass1_pred}")
    sep = "\n\n" if variant.spacing in ("roomy", "spacious") else "\n"
    return sep.join([b for b in blocks if b and str(b).strip()])

# ============================================================
# GENERATION / LOGITS
# ============================================================

@torch.inference_mode()
def generate(prompt, max_new_tokens, temperature=TEMPERATURE, top_p=TOP_P):
    inputs = TOKENIZER(prompt, return_tensors="pt").to(DEVICE)
    
    gen_kwargs = {
        "max_new_tokens": max_new_tokens,
        "pad_token_id": TOKENIZER.eos_token_id,
        "eos_token_id": TOKENIZER.eos_token_id,
    }
    
    if temperature > 0:
        gen_kwargs["do_sample"] = True
        gen_kwargs["temperature"] = temperature
        gen_kwargs["top_p"] = top_p
    else:
        gen_kwargs["do_sample"] = False
        
    out = MODEL.generate(
        **inputs,
        **gen_kwargs
    )
    
    full_output = TOKENIZER.decode(out[0], skip_special_tokens=True)
    prompt_len = inputs["input_ids"].shape[1]
    completion = TOKENIZER.decode(out[0][prompt_len:], skip_special_tokens=True)
    return full_output, completion


def prompt_logit_analysis(prompt, target_letter: str, candidate_groups: List[List[int]]):
    inputs = TOKENIZER(prompt, return_tensors="pt").to(DEVICE)
    with torch.inference_mode():
        out = MODEL(
            **inputs,
            output_hidden_states=True,
            output_attentions=True,
            use_cache=False,
            return_dict=True,
        )

    logits = out.logits[0, -1].float()
    probs = torch.softmax(logits, dim=-1)
    log_probs = torch.log_softmax(logits, dim=-1)
    full_entropy = float((-probs * log_probs).sum().item())

    ids = token_ids_for_text(TOKENIZER, target_letter)
    tid, tlogit = target_from_ids(logits, ids)
    if tid is None:
        tid = int(TOKENIZER.eos_token_id)
        tlogit = float(logits[tid].item())
    target_prob = float(probs[tid].item())
    target_logprob = float(log_probs[tid].item())

    masked = logits.clone()
    masked[ids] = float("-inf")
    margin = float(tlogit - masked.max().item())
    rank = int((logits > tlogit).sum().item() + 1)

    cand = candidate_group_metrics(logits, candidate_groups, 0 if target_letter == "yes" else 1) if candidate_groups else None

    # DLA proxy.
    target_weight = LM_HEAD.weight[tid].float()
    dla_curve = []
    for h in out.hidden_states:
        vec = h[:, -1, :]
        if FINAL_NORM is not None:
            try:
                vec = FINAL_NORM(vec)
            except Exception:
                pass
        dla_curve.append(float(torch.dot(vec.squeeze(0).float(), target_weight).item()))

    # Prefix-attention mass.
    attn_curve = []
    for layer_attn in out.attentions:
        a = layer_attn[0]
        qpos = a.shape[-2] - 1
        prefix_idx = list(range(min(8, a.shape[-1])))
        if len(prefix_idx) == 0:
            attn_curve.append(float("nan"))
        else:
            attn_curve.append(float(a[:, qpos, prefix_idx].sum(dim=-1).mean().item()))

    top_probs, top_ids = torch.topk(probs, k=min(6, probs.shape[-1]))
    top_tokens = TOKENIZER.convert_ids_to_tokens(top_ids.tolist())
    top_snapshot = [{"token": t, "id": int(i), "prob": float(p)} for t, i, p in zip(top_tokens, top_ids.tolist(), top_probs.tolist())]

    return {
        "target_id": int(tid),
        "target_token": TOKENIZER.convert_ids_to_tokens([int(tid)])[0],
        "target_logit": float(tlogit),
        "target_logprob": float(target_logprob),
        "target_prob": float(target_prob),
        "target_margin": float(margin),
        "target_rank": int(rank),
        "full_entropy": float(full_entropy),
        "candidate_metrics": cand,
        "top_snapshot": top_snapshot,
        "dla_curve": [float(x) for x in dla_curve],
        "attention_curve": [float(x) for x in attn_curve],
        "n_prompt_tokens": int(inputs["input_ids"].shape[1]),
    }

# ============================================================
# QUESTION-LEVEL HELPERS
# ============================================================

def transition_state(p1_correct, p2_correct):
    return f"{'correct' if p1_correct else 'wrong'}→{'correct' if p2_correct else 'wrong'}"


def basin_class(stability, entropy):
    if stability >= 0.75 and entropy <= 1.0:
        return "stable correct basin"
    if stability >= 0.75 and entropy > 1.0:
        return "stable but diverse basin"
    if stability <= 0.45 and entropy > 1.5:
        return "flip-prone basin"
    if stability <= 0.55:
        return "uncertain basin"
    return "moderately stable basin"

# ============================================================
# CALIBRATION
# ============================================================

def calibration_threshold(scores, labels):
    scores = np.asarray(scores, dtype=np.float64)
    labels = np.asarray(labels, dtype=np.int64)
    thresholds = np.quantile(scores, np.linspace(0.02, 0.98, N_THRESHOLD_STEPS))
    rows = []
    best = {"threshold": 0.0, "accuracy": -1.0}
    for t in thresholds:
        preds = (scores >= t).astype(np.int64)
        acc = float((preds == labels).mean())
        rows.append({"threshold": float(t), "accuracy": acc})
        if acc > best["accuracy"]:
            best = {"threshold": float(t), "accuracy": acc}
    return best, pd.DataFrame(rows)

# ============================================================
# RUN A SINGLE SAMPLE THROUGH ALL VARIANTS
# ============================================================

def run_sample(sample):
    q = sample["question"]
    gold = sample["gold"]
    qid = sample["question_id"]
    rows = []

    for variant in VARIANTS:
        prompt1 = build_pass1_prompt(q, variant)
        full1, comp1 = generate(prompt1, PASS1_MAX_NEW)
        p1_pred = extract_yes_no(comp1)

        p1_metrics = prompt_logit_analysis(
            prompt1,
            gold,
            candidate_groups=[token_ids_for_text(TOKENIZER, "yes"), token_ids_for_text(TOKENIZER, "no")],
        )
        p1_probs = p1_metrics["candidate_metrics"]["probs"]
        p1_logodds = math.log((p1_probs[0] + 1e-12) / (p1_probs[1] + 1e-12))
        p1_commit = commitment_score(comp1)
        p1_correct = (p1_pred == gold)
        p1_conf = 1.0 / (1.0 + np.exp(-p1_logodds))

        prompt2 = build_pass2_prompt(q, variant, comp1, p1_pred)
        full2, comp2 = generate(prompt2, PASS2_MAX_NEW)
        p2_pred = extract_yes_no(comp2)

        p2_metrics = prompt_logit_analysis(
            prompt2,
            gold,
            candidate_groups=[token_ids_for_text(TOKENIZER, "yes"), token_ids_for_text(TOKENIZER, "no")],
        )
        p2_probs = p2_metrics["candidate_metrics"]["probs"]
        p2_logodds = math.log((p2_probs[0] + 1e-12) / (p2_probs[1] + 1e-12))
        p2_commit = commitment_score(comp2)
        p2_correct = (p2_pred == gold)
        p2_conf = 1.0 / (1.0 + np.exp(-p2_logodds))

        # calibration placeholders; filled later using calibrated thresholds.
        row = {
            "question_id": qid,
            "question": q,
            "gold": gold,
            "variant": variant.name,
            "strength": variant.strength,
            "note": variant.note,
            "pass1_prompt": prompt1,
            "pass1_full_output": full1,
            "pass1_completion": comp1,
            "pass1_prediction": p1_pred,
            "pass1_correct": bool(p1_correct),
            "pass1_logodds": float(p1_logodds),
            "pass1_confidence": float(p1_conf),
            "pass1_commitment": float(p1_commit),
            "pass1_yes_prob": float(p1_probs[0]),
            "pass1_no_prob": float(p1_probs[1]),
            "pass1_target_prob": float(p1_metrics["candidate_metrics"]["target_prob"]),
            "pass1_margin": float(p1_metrics["candidate_metrics"]["margin"]),
            "pass1_entropy": float(p1_metrics["candidate_metrics"]["entropy"]),
            "pass1_flip_uncertainty": float(p1_metrics["candidate_metrics"]["flip_uncertainty"]),
            "pass1_target_logprob": float(p1_metrics["target_logprob"]),
            "pass1_target_rank": int(p1_metrics["target_rank"]),
            "pass1_target_token": p1_metrics["target_token"],
            "pass1_full_entropy": float(p1_metrics["full_entropy"]),
            "pass1_top_snapshot": to_jsonable(p1_metrics["top_snapshot"]),
            "pass1_dla_curve": to_jsonable(p1_metrics["dla_curve"]),
            "pass1_attention_curve": to_jsonable(p1_metrics["attention_curve"]),
            "pass1_prompt_tokens": int(p1_metrics["n_prompt_tokens"]),
            "pass2_prompt": prompt2,
            "pass2_full_output": full2,
            "pass2_completion": comp2,
            "pass2_prediction": p2_pred,
            "pass2_correct": bool(p2_correct),
            "pass2_logodds": float(p2_logodds),
            "pass2_confidence": float(p2_conf),
            "pass2_commitment": float(p2_commit),
            "pass2_yes_prob": float(p2_probs[0]),
            "pass2_no_prob": float(p2_probs[1]),
            "pass2_target_prob": float(p2_metrics["candidate_metrics"]["target_prob"]),
            "pass2_margin": float(p2_metrics["candidate_metrics"]["margin"]),
            "pass2_entropy": float(p2_metrics["candidate_metrics"]["entropy"]),
            "pass2_flip_uncertainty": float(p2_metrics["candidate_metrics"]["flip_uncertainty"]),
            "pass2_target_logprob": float(p2_metrics["target_logprob"]),
            "pass2_target_rank": int(p2_metrics["target_rank"]),
            "pass2_target_token": p2_metrics["target_token"],
            "pass2_full_entropy": float(p2_metrics["full_entropy"]),
            "pass2_top_snapshot": to_jsonable(p2_metrics["top_snapshot"]),
            "pass2_dla_curve": to_jsonable(p2_metrics["dla_curve"]),
            "pass2_attention_curve": to_jsonable(p2_metrics["attention_curve"]),
            "pass2_prompt_tokens": int(p2_metrics["n_prompt_tokens"]),
            "transition": transition_state(p1_correct, p2_correct),
            "improved": bool((not p1_correct) and p2_correct),
            "regressed": bool(p1_correct and (not p2_correct)),
            "unchanged": bool(p1_correct == p2_correct),
            "pass1_vs_pass2_prediction_changed": bool(p1_pred != p2_pred),
            "pass1_vs_pass2_text_changed": bool(normalize_text(comp1) != normalize_text(comp2)),
            "pass1_vs_pass2_text_similarity": float(sequence_similarity(comp1, comp2)),
            "pass1_vs_pass2_jaccard_similarity": float(jaccard_token_similarity(comp1, comp2)),
            "response_drift_sequence": float(1.0 - sequence_similarity(comp1, comp2)),
            "delta_logodds": float(p2_logodds - p1_logodds),
            "delta_confidence": float(p2_conf - p1_conf),
            "delta_commitment": float(p2_commit - p1_commit),
            "delta_margin": float(p2_metrics["candidate_metrics"]["margin"] - p1_metrics["candidate_metrics"]["margin"]),
            "delta_entropy": float(p2_metrics["candidate_metrics"]["entropy"] - p1_metrics["candidate_metrics"]["entropy"]),
            "delta_flip_uncertainty": float(p2_metrics["candidate_metrics"]["flip_uncertainty"] - p1_metrics["candidate_metrics"]["flip_uncertainty"]),
        }
        rows.append(row)

    return rows

# ============================================================
# APPLY CALIBRATION THRESHOLDS
# ============================================================

def apply_thresholds(df, thresholds):
    out = df.copy()
    p1_cal_pred = []
    p2_cal_pred = []
    p1_cal_correct = []
    p2_cal_correct = []
    p1_cal_conf = []
    p2_cal_conf = []
    for _, r in out.iterrows():
        thr = thresholds[r["variant"]]
        p1 = "yes" if float(r["pass1_logodds"]) >= thr["pass1"] else "no"
        p2 = "yes" if float(r["pass2_logodds"]) >= thr["pass2"] else "no"
        p1_cal_pred.append(p1)
        p2_cal_pred.append(p2)
        p1_cal_correct.append(p1 == r["gold"])
        p2_cal_correct.append(p2 == r["gold"])
        p1_cal_conf.append(float(1.0 / (1.0 + np.exp(-float(r["pass1_logodds"])))))
        p2_cal_conf.append(float(1.0 / (1.0 + np.exp(-float(r["pass2_logodds"])))))
        
    out["pass1_calibrated_prediction"] = p1_cal_pred
    out["pass2_calibrated_prediction"] = p2_cal_pred
    out["pass1_calibrated_correct"] = p1_cal_correct
    out["pass2_calibrated_correct"] = p2_cal_correct
    out["pass1_calibrated_confidence"] = p1_cal_conf
    out["pass2_calibrated_confidence"] = p2_cal_conf
    return out

# ============================================================
# SUMMARY METRICS
# ============================================================

def transition_matrix(df):
    mat = pd.DataFrame(0, index=["wrong→wrong", "wrong→correct", "correct→wrong", "correct→correct"], columns=["count"])
    counts = Counter(df["transition"].tolist())
    for k in mat.index:
        mat.loc[k, "count"] = counts.get(k, 0)
    return mat


def summarize_variant(df):
    rows = []
    for variant, g in df.groupby("variant"):
        counts = Counter(g["transition"].tolist())
        total = max(1, len(g))
        q_stats = g.groupby("question_id", as_index=False).agg(
            stability=("pass1_vs_pass2_prediction_changed", lambda x: 1.0 - float(np.mean(x))),
            improv=("improved", "mean"),
            regress=("regressed", "mean"),
            drift=("response_drift_sequence", "mean"),
        )
        rows.append({
            "variant": variant,
            "strength": float(g["strength"].iloc[0]),
            "n_rows": int(len(g)),
            "pass1_raw_accuracy": float(g["pass1_correct"].mean()),
            "pass2_raw_accuracy": float(g["pass2_correct"].mean()),
            "pass1_cal_accuracy": float(g["pass1_calibrated_correct"].mean()),
            "pass2_cal_accuracy": float(g["pass2_calibrated_correct"].mean()),
            "pass1_commitment_mean": float(g["pass1_commitment"].mean()),
            "pass2_commitment_mean": float(g["pass2_commitment"].mean()),
            "pass1_logodds_mean": float(g["pass1_logodds"].mean()),
            "pass2_logodds_mean": float(g["pass2_logodds"].mean()),
            "pass1_entropy_mean": float(g["pass1_entropy"].mean()),
            "pass2_entropy_mean": float(g["pass2_entropy"].mean()),
            "pass1_margin_mean": float(g["pass1_margin"].mean()),
            "pass2_margin_mean": float(g["pass2_margin"].mean()),
            "improvement_rate": float(g["improved"].mean()),
            "regression_rate": float(g["regressed"].mean()),
            "prediction_change_rate": float(g["pass1_vs_pass2_prediction_changed"].mean()),
            "text_change_rate": float(g["pass1_vs_pass2_text_changed"].mean()),
            "mean_text_similarity": float(g["pass1_vs_pass2_text_similarity"].mean()),
            "mean_jaccard_similarity": float(g["pass1_vs_pass2_jaccard_similarity"].mean()),
            "mean_response_drift": float(g["response_drift_sequence"].mean()),
            "mean_delta_logodds": float(g["delta_logodds"].mean()),
            "mean_delta_commitment": float(g["delta_commitment"].mean()),
            "mean_delta_margin": float(g["delta_margin"].mean()),
            "mean_delta_entropy": float(g["delta_entropy"].mean()),
            "mean_flip_uncertainty_delta": float(g["delta_flip_uncertainty"].mean()),
            "transition_entropy": float(entropy_from_probs(np.array(list(counts.values()), dtype=np.float64) / total)),
            "q_stability_mean": float(q_stats["stability"].mean()),
            "q_improvement_mean": float(q_stats["improv"].mean()),
            "q_regression_mean": float(q_stats["regress"].mean()),
            "q_drift_mean": float(q_stats["drift"].mean()),
        })
    return pd.DataFrame(rows)


def summarize_question_level(df):
    rows = []
    for qid, g in df.groupby("question_id"):
        transition_counts = Counter(g["transition"].tolist())
        pass1_counts = Counter(g["pass1_prediction"].dropna().tolist())
        pass2_counts = Counter(g["pass2_prediction"].dropna().tolist())
        p1_majority = pass1_counts.most_common(1)[0][0] if pass1_counts else None
        p2_majority = pass2_counts.most_common(1)[0][0] if pass2_counts else None
        rows.append({
            "question_id": qid,
            "question": g["question"].iloc[0],
            "gold": g["gold"].iloc[0],
            "n_variants": int(len(g)),
            "pass1_majority": p1_majority,
            "pass2_majority": p2_majority,
            "pass1_accuracy": float(g["pass1_correct"].mean()),
            "pass2_accuracy": float(g["pass2_correct"].mean()),
            "pass1_cal_accuracy": float(g["pass1_calibrated_correct"].mean()),
            "pass2_cal_accuracy": float(g["pass2_calibrated_correct"].mean()),
            "stability_score": float(1.0 - g["pass1_vs_pass2_prediction_changed"].mean()),
            "transition_entropy": float(entropy_from_probs(np.array(list(transition_counts.values()), dtype=np.float64) / max(1, len(g)))),
            "improvement_rate": float(g["improved"].mean()),
            "regression_rate": float(g["regressed"].mean()),
            "pass1_logodds_mean": float(g["pass1_logodds"].mean()),
            "pass2_logodds_mean": float(g["pass2_logodds"].mean()),
            "pass1_commitment_mean": float(g["pass1_commitment"].mean()),
            "pass2_commitment_mean": float(g["pass2_commitment"].mean()),
            "response_drift_mean": float(g["response_drift_sequence"].mean()),
            "delta_logodds_mean": float(g["delta_logodds"].mean()),
            "delta_commitment_mean": float(g["delta_commitment"].mean()),
            "delta_margin_mean": float(g["delta_margin"].mean()),
            "delta_entropy_mean": float(g["delta_entropy"].mean()),
            "basin_class": basin_class(float(1.0 - g["pass1_vs_pass2_prediction_changed"].mean()), float(entropy_from_probs(np.array(list(transition_counts.values()), dtype=np.float64) / max(1, len(g)))))
        })
    qdf = pd.DataFrame(rows)
    return qdf


def summarize_question_variant_grid(df):
    rows = []
    for (qid, variant), g in df.groupby(["question_id", "variant"]):
        rows.append({
            "question_id": qid,
            "variant": variant,
            "strength": float(g["strength"].iloc[0]),
            "gold": g["gold"].iloc[0],
            "question": g["question"].iloc[0],
            "pass1_correct": float(g["pass1_correct"].mean()),
            "pass2_correct": float(g["pass2_correct"].mean()),
            "pass1_cal_correct": float(g["pass1_calibrated_correct"].mean()),
            "pass2_cal_correct": float(g["pass2_calibrated_correct"].mean()),
            "pass1_logodds": float(g["pass1_logodds"].mean()),
            "pass2_logodds": float(g["pass2_logodds"].mean()),
            "pass1_commitment": float(g["pass1_commitment"].mean()),
            "pass2_commitment": float(g["pass2_commitment"].mean()),
            "delta_logodds": float(g["delta_logodds"].mean()),
            "delta_commitment": float(g["delta_commitment"].mean()),
            "delta_margin": float(g["delta_margin"].mean()),
            "response_drift": float(g["response_drift_sequence"].mean()),
            "transition_entropy": float(entropy_from_probs(np.array(list(Counter(g["transition"]).values()), dtype=np.float64) / max(1, len(g)))),
            "improvement_rate": float(g["improved"].mean()),
            "regression_rate": float(g["regressed"].mean()),
            "prediction_change_rate": float(g["pass1_vs_pass2_prediction_changed"].mean()),
            "text_change_rate": float(g["pass1_vs_pass2_text_changed"].mean()),
            "transition_majority": Counter(g["transition"].tolist()).most_common(1)[0][0] if len(g) else None,
        })
    return pd.DataFrame(rows)

# ============================================================
# PLOTTING / DASHBOARDS
# ============================================================

def plot_reliability(df, variant, pass_col, path, bins=N_RELIABILITY_BINS):
    scores = np.asarray(df[f"{pass_col}_logodds"].tolist(), dtype=np.float64)
    accs = np.asarray(df[f"{pass_col}_correct"].astype(int).tolist(), dtype=np.int64)
    conf = 1.0 / (1.0 + np.exp(-scores))
    bin_edges = np.linspace(0, 1, bins + 1)
    bin_ids = np.clip(np.digitize(conf, bin_edges) - 1, 0, bins - 1)
    rows = []
    for b in range(bins):
        mask = bin_ids == b
        if mask.sum() == 0:
            rows.append({"bin": b, "avg_conf": np.nan, "accuracy": np.nan, "count": 0})
        else:
            rows.append({
                "bin": b,
                "avg_conf": float(conf[mask].mean()),
                "accuracy": float(accs[mask].mean()),
                "count": int(mask.sum()),
            })
    r = pd.DataFrame(rows)
    r.dropna(inplace=True)
    if len(r) == 0:
        return pd.DataFrame(rows)
    plt.figure(figsize=(6.7, 5.6))
    plt.plot([0, 1], [0, 1], linestyle="--", color="black", alpha=0.6, label="ideal")
    plt.plot(r["avg_conf"], r["accuracy"], marker="o", linewidth=2, label=variant)
    plt.fill_between(r["avg_conf"], np.maximum(0, r["accuracy"] - 0.05), np.minimum(1, r["accuracy"] + 0.05), alpha=0.12)
    plt.xlabel("Commitment-derived confidence")
    plt.ylabel("Accuracy")
    plt.title(f"Reliability diagram ({variant}, {pass_col})")
    plt.legend(frameon=True)
    plt.tight_layout()
    plt.savefig(path, dpi=180)
    plt.close()
    return r


def plot_bifurcation(df, path):
    agg = df.groupby("variant", as_index=False).agg(
        strength=("strength", "mean"),
        pass1_acc=("pass1_correct", "mean"),
        pass2_acc=("pass2_correct", "mean"),
        pass1_logodds=("pass1_logodds", "mean"),
        pass2_logodds=("pass2_logodds", "mean"),
        pass1_commit=("pass1_commitment", "mean"),
        pass2_commit=("pass2_commitment", "mean"),
        transition_entropy=("transition_entropy", "mean"),
    ).sort_values("strength")
    fig, axes = plt.subplots(2, 2, figsize=(16, 11), constrained_layout=True)
    panels = [
        ("pass1_acc", "Pass-1 accuracy"),
        ("pass2_acc", "Pass-2 accuracy"),
        ("pass2_logodds", "Pass-2 log-odds"),
        ("pass2_commit", "Pass-2 commitment"),
    ]
    for ax, (col, lab) in zip(axes.ravel(), panels):
        x = agg["strength"].to_numpy(dtype=np.float64)
        y = agg[col].to_numpy(dtype=np.float64)
        ax.scatter(x, y, s=65)
        xs, ys = polyfit_line(x, y, deg=2)
        if xs is not None:
            ax.plot(xs, ys, linewidth=2)
        for _, r in agg.iterrows():
            ax.annotate(r["variant"], (r["strength"], r[col]), fontsize=8, xytext=(3, 3), textcoords="offset points")
        ax.set_xlabel("Prompt strength")
        ax.set_ylabel(lab)
        ax.set_title(f"{lab} vs prompt strength")
    plt.savefig(path, dpi=180, bbox_inches="tight")
    plt.close()


def plot_phase_portrait(df, path):
    fig, axes = plt.subplots(1, 2, figsize=(14.5, 5.8), constrained_layout=True)
    cmap = {"correct→correct": "tab:green", "wrong→correct": "tab:blue", "correct→wrong": "tab:red", "wrong→wrong": "tab:gray"}
    for ax, pass_col in zip(axes, ["pass1", "pass2"]):
        colors = df["transition"].map(cmap).tolist()
        ax.scatter(df[f"{pass_col}_logodds"], df[f"{pass_col}_commitment"], c=colors, alpha=0.72, s=20)
        ax.axvline(0, color="black", linewidth=1, alpha=0.35)
        ax.axhline(0.5, color="black", linewidth=1, alpha=0.2)
        ax.set_title(f"Phase portrait ({pass_col})")
        ax.set_xlabel("Log-odds")
        ax.set_ylabel("Commitment")
    plt.savefig(path, dpi=180)
    plt.close()


def plot_transition_heatmap(df, path):
    counts = Counter(df["transition"].tolist())
    mat = np.array([[
        counts.get("wrong→wrong", 0), counts.get("wrong→correct", 0)
    ], [
        counts.get("correct→wrong", 0), counts.get("correct→correct", 0)
    ]], dtype=np.float64)
    heatmap(mat, ["wrong→wrong", "wrong→correct"], ["wrong→wrong", "correct→wrong"], "Transition matrix (counts)", path, xlabel="Pass-2 outcome", ylabel="Pass-1 outcome", cmap="Blues", fmt="{:.0f}")


def plot_delta_heatmap(df, xcol, ycol, zcol, title, path, xbins=8, ybins=8):
    sub = df.copy()
    x = sub[xcol].to_numpy(dtype=np.float64)
    y = sub[ycol].to_numpy(dtype=np.float64)
    z = sub[zcol].astype(float).to_numpy(dtype=np.float64)
    x_edges = np.unique(np.quantile(x, np.linspace(0, 1, xbins + 1)))
    y_edges = np.unique(np.quantile(y, np.linspace(0, 1, ybins + 1)))
    if len(x_edges) < 3 or len(y_edges) < 3:
        return None
    xi = np.clip(np.digitize(x, x_edges) - 1, 0, len(x_edges) - 2)
    yi = np.clip(np.digitize(y, y_edges) - 1, 0, len(y_edges) - 2)
    mat = np.full((len(y_edges) - 1, len(x_edges) - 1), np.nan, dtype=np.float64)
    for i in range(len(y_edges) - 1):
        for j in range(len(x_edges) - 1):
            mask = (xi == j) & (yi == i)
            if mask.sum() > 0:
                mat[i, j] = float(z[mask].mean())
    plt.figure(figsize=(9.4, 6.8))
    im = plt.imshow(mat, aspect="auto", origin="lower", cmap="coolwarm",
                    norm=TwoSlopeNorm(vmin=np.nanmin(mat), vcenter=0.5, vmax=np.nanmax(mat)))
    plt.colorbar(im)
    plt.title(title)
    plt.xlabel(xcol)
    plt.ylabel(ycol)
    plt.tight_layout()
    plt.savefig(path, dpi=180)
    plt.close()
    return mat


def plot_mean_trajectory(df, path):
    fig, ax = plt.subplots(figsize=(8.9, 6.4))
    cmap = {"correct→correct": "tab:green", "wrong→correct": "tab:blue", "correct→wrong": "tab:red", "wrong→wrong": "tab:gray"}
    for trans, g in df.groupby("transition"):
        ax.scatter(g["pass1_logodds"], g["pass1_commitment"], s=22, alpha=0.6, color=cmap.get(trans, "gray"), label=trans)
        dx = (g["pass2_logodds"] - g["pass1_logodds"]).to_numpy(dtype=np.float64)
        dy = (g["pass2_commitment"] - g["pass1_commitment"]).to_numpy(dtype=np.float64)
        ax.quiver(g["pass1_logodds"].to_numpy(dtype=np.float64)[::QUIVER_STEP], g["pass1_commitment"].to_numpy(dtype=np.float64)[::QUIVER_STEP],
                  dx[::QUIVER_STEP], dy[::QUIVER_STEP], angles="xy", scale_units="xy", scale=1.0, alpha=0.25, width=0.002)
    ax.axvline(0, color="black", linewidth=1, alpha=0.35)
    ax.set_xlabel("Pass-1 log-odds")
    ax.set_ylabel("Pass-1 commitment")
    ax.set_title("Decision trajectory field")
    ax.legend(frameon=True, fontsize=8)
    plt.tight_layout()
    plt.savefig(path, dpi=180)
    plt.close()


def plot_variant_metric_dashboard(summary_df, path):
    sdf = summary_df.sort_values("strength")
    fig, axes = plt.subplots(2, 3, figsize=(18, 11), constrained_layout=True)
    panels = [
        ("pass1_raw_accuracy", "Pass-1 raw accuracy"),
        ("pass2_raw_accuracy", "Pass-2 raw accuracy"),
        ("improvement_rate", "Improvement rate"),
        ("regression_rate", "Regression rate"),
        ("pass1_commitment_mean", "Pass-1 commitment"),
        ("pass2_commitment_mean", "Pass-2 commitment"),
    ]
    for ax, (metric, title) in zip(axes.ravel(), panels):
        ax.bar(sdf["variant"], sdf[metric], color=plt.get_cmap("tab10")(np.linspace(0.1, 0.9, len(sdf))))
        ax.set_title(title)
        ax.tick_params(axis="x", rotation=30)
        annotated_bar(ax)
    plt.savefig(path, dpi=180, bbox_inches="tight")
    plt.close()


def plot_question_glyph_atlas(df, path, top_n=12):
    qdf = df.groupby("question_id", as_index=False).agg(
        stability=("stability_score", "mean"),
        entropy=("transition_entropy", "mean"),
        improvement=("improvement_rate", "mean"),
        regression=("regression_rate", "mean"),
        pass1_acc=("pass1_correct", "mean"),
        pass2_acc=("pass2_correct", "mean"),
        question=("question", "first"),
    )
    if len(qdf) == 0:
        return
    qdf = qdf.sort_values(["stability", "entropy"], ascending=[True, False]).head(top_n)
    rows = len(qdf)
    fig, axes = plt.subplots(rows, 1, figsize=(13.5, max(4.5, rows * 1.2)), constrained_layout=True)
    if rows == 1:
        axes = [axes]
    for ax, (_, r) in zip(axes, qdf.iterrows()):
        sub = df[df["question_id"] == r["question_id"]]
        counts = sub["pass2_prediction"].value_counts().reindex(["yes", "no"]).fillna(0).astype(int)
        vals = counts.values.reshape(1, -1)
        im = ax.imshow(vals, aspect="auto", cmap="magma", vmin=0, vmax=max(1, vals.max()))
        ax.set_yticks([])
        ax.set_xticks(range(2))
        ax.set_xticklabels(["yes", "no"])
        ax.set_title(f"Q{r['question_id']} | stab={r['stability']:.2f} | ent={r['entropy']:.2f} | imp={r['improvement']:.2f} | reg={r['regression']:.2f} | {preview_text(r['question'], 80)}")
        for j in range(2):
            ax.text(j, 0, str(int(vals[0, j])), ha="center", va="center", color="white" if vals[0, j] > vals.max() * 0.4 else "black", fontsize=9)
    plt.suptitle("Question-level glyph atlas", fontsize=16, y=1.01)
    plt.savefig(path, dpi=180, bbox_inches="tight")
    plt.close()


def plot_basin_heatmaps(df, out_prefix):
    for zcol, title, fname in [
        ("improved", "Improvement basin", f"{out_prefix}_improvement_basin.png"),
        ("regressed", "Regression basin", f"{out_prefix}_regression_basin.png"),
        ("pass2_correct", "Pass-2 correctness basin", f"{out_prefix}_pass2_correct_basin.png"),
        ("transition_entropy", "Transition entropy basin", f"{out_prefix}_transition_entropy_basin.png"),
    ]:
        plot_delta_heatmap(
            df,
            xcol="pass1_logodds",
            ycol="pass1_commitment",
            zcol=zcol,
            title=title,
            path=os.path.join(PLOTS_DIR, fname),
            xbins=8,
            ybins=8,
        )


def plot_representative_trajectories(df, path):
    qdf = df.groupby("question_id", as_index=False).agg(
        stability=("stability_score", "mean"),
        entropy=("transition_entropy", "mean"),
        improvement=("improvement_rate", "mean"),
        regression=("regression_rate", "mean"),
        pass1_acc=("pass1_correct", "mean"),
        pass2_acc=("pass2_correct", "mean"),
    )
    ids = []
    if len(qdf) > 0:
        ids.extend(qdf.sort_values(["stability", "pass1_acc"], ascending=[False, False]).head(1)["question_id"].tolist())
        ids.extend(qdf.sort_values(["stability", "pass1_acc"], ascending=[True, True]).head(1)["question_id"].tolist())
        ids.extend(qdf.sort_values(["improvement", "stability"], ascending=[False, True]).head(1)["question_id"].tolist())
        ids.extend(qdf.sort_values(["regression", "stability"], ascending=[False, True]).head(1)["question_id"].tolist())
        ids.extend(qdf.sort_values(["entropy", "stability"], ascending=[False, True]).head(1)["question_id"].tolist())
        ids = list(dict.fromkeys(ids))[:REPRESENTATIVE_QUESTIONS]
    if len(ids) == 0:
        return

    rows = len(ids)
    fig, axes = plt.subplots(rows, 2, figsize=(16, 4.4 * rows), constrained_layout=True)
    if rows == 1:
        axes = np.array([axes])
    for ridx, qid in enumerate(ids):
        sdf = df[df["question_id"] == qid].copy()
        sdf = sdf.sort_values("strength")
        ax1 = axes[ridx, 0]
        ax2 = axes[ridx, 1]
        ax1.plot(sdf["strength"], sdf["pass1_logodds"], marker="o", label="pass1", linewidth=2)
        ax1.plot(sdf["strength"], sdf["pass2_logodds"], marker="s", label="pass2", linewidth=2)
        ax1.axhline(0, color="black", linewidth=1, alpha=0.35)
        ax1.set_title(f"Q{qid} | {preview_text(sdf['question'].iloc[0], 74)}")
        ax1.set_xlabel("Prompt strength")
        ax1.set_ylabel("Log-odds")
        ax1.legend(frameon=True)

        ax2.plot(sdf["strength"], sdf["pass1_commitment"], marker="o", label="pass1", linewidth=2)
        ax2.plot(sdf["strength"], sdf["pass2_commitment"], marker="s", label="pass2", linewidth=2)
        ax2.set_title(f"Q{qid} commitment trajectory")
        ax2.set_xlabel("Prompt strength")
        ax2.set_ylabel("Commitment")
        ax2.legend(frameon=True)

    plt.suptitle("Representative stability trajectories", fontsize=16, y=1.01)
    plt.savefig(path, dpi=180, bbox_inches="tight")
    plt.close()


def plot_basin_map(df, path):
    fig, axes = plt.subplots(2, 2, figsize=(16, 11), constrained_layout=True)
    cmap = {"correct→correct": "tab:green", "wrong→correct": "tab:blue", "correct→wrong": "tab:red", "wrong→wrong": "tab:gray"}
    
    axes[0, 0].scatter(df["pass1_logodds"], df["pass1_commitment"], c=df["transition"].map(cmap), alpha=0.72, s=18)
    axes[0, 0].axvline(0, color="black", linewidth=1, alpha=0.35)
    axes[0, 0].set_title("Pass-1 basin phase portrait")
    axes[0, 0].set_xlabel("Log-odds")
    axes[0, 0].set_ylabel("Commitment")

    axes[0, 1].scatter(df["pass2_logodds"], df["pass2_commitment"], c=df["transition"].map(cmap), alpha=0.72, s=18)
    axes[0, 1].axvline(0, color="black", linewidth=1, alpha=0.35)
    axes[0, 1].set_title("Pass-2 basin phase portrait")
    axes[0, 1].set_xlabel("Log-odds")
    axes[0, 1].set_ylabel("Commitment")

    counts = Counter(df["transition"].tolist())
    mat = np.array([
        [counts.get("wrong→wrong", 0), counts.get("wrong→correct", 0)],
        [counts.get("correct→wrong", 0), counts.get("correct→correct", 0)],
    ], dtype=np.float64)
    im = axes[1, 0].imshow(mat, aspect="auto", cmap="Blues")
    axes[1, 0].set_title("Transition matrix")
    axes[1, 0].set_xticks([0, 1])
    axes[1, 0].set_xticklabels(["W→W", "W→C"])
    axes[1, 0].set_yticks([0, 1])
    axes[1, 0].set_yticklabels(["W→W", "C→W"])
    for i in range(2):
        for j in range(2):
            axes[1, 0].text(j, i, str(int(mat[i, j])), ha="center", va="center", fontsize=10)
    fig.colorbar(im, ax=axes[1, 0], fraction=0.046, pad=0.04)

    sub = df.iloc[::QUIVER_STEP].copy()
    axes[1, 1].scatter(sub["pass1_logodds"], sub["pass1_commitment"], c=sub["transition"].map(cmap), alpha=0.6, s=18)
    axes[1, 1].quiver(sub["pass1_logodds"], sub["pass1_commitment"], sub["delta_logodds"], sub["delta_commitment"],
                      angles="xy", scale_units="xy", scale=1.0, alpha=0.3, width=0.002)
    axes[1, 1].axvline(0, color="black", linewidth=1, alpha=0.35)
    axes[1, 1].axhline(0.5, color="black", linewidth=1, alpha=0.2)
    axes[1, 1].set_title("Decision vector field")
    axes[1, 1].set_xlabel("Pass-1 log-odds")
    axes[1, 1].set_ylabel("Pass-1 commitment")

    plt.suptitle("Self-correction basin atlas", fontsize=16, y=1.01)
    plt.savefig(path, dpi=180, bbox_inches="tight")
    plt.close()


# ============================================================
# MAIN EXPERIMENT
# ============================================================

def calibrate_thresholds(validation_samples):
    thresholds = {}
    threshold_rows = []
    for variant in VARIANTS:
        thresholds[variant.name] = {}
        for pass_id in [1, 2]:
            scores = []
            labels = []
            for sample in validation_samples:
                q = sample["question"]
                gold = sample["gold"]
                prompt1 = build_pass1_prompt(q, variant)
                full1, comp1 = generate(prompt1, PASS1_MAX_NEW)
                p1_pred = extract_yes_no(comp1)
                if pass_id == 1:
                    prompt = prompt1
                else:
                    prompt = build_pass2_prompt(q, variant, comp1, p1_pred)
                metrics = prompt_logit_analysis(
                    prompt,
                    gold,
                    candidate_groups=[token_ids_for_text(TOKENIZER, "yes"), token_ids_for_text(TOKENIZER, "no")],
                )
                logodds = math.log((metrics["candidate_metrics"]["probs"][0] + 1e-12) / (metrics["candidate_metrics"]["probs"][1] + 1e-12))
                scores.append(logodds)
                labels.append(1 if gold == "yes" else 0)
                free_memory()
            best, curve_df = calibration_threshold(scores, labels)
            thresholds[variant.name][f"pass{pass_id}"] = best["threshold"]
            curve_df["variant"] = variant.name
            curve_df["pass_id"] = pass_id
            threshold_rows.append(curve_df)
    return thresholds, pd.concat(threshold_rows, ignore_index=True)


def main():
    print("Loading model and tokenizer ...")
    load_model()

    print("Loading validation / test splits ...")
    # Changed "validation" to "train" to match the available splits in the HF repo
    val_samples = load_strategyqa_split("train", N_VALIDATION, "strategyqa_train_basin")
    test_samples = load_strategyqa_split("test", N_TEST, "strategyqa_test_basin")

    print("Calibrating thresholds on validation split ...")
    thresholds, threshold_df = calibrate_thresholds(val_samples)
    save_df(threshold_df, os.path.join(CSV_DIR, "threshold_search.csv"))
    save_json(thresholds, os.path.join(REPORTS_DIR, "thresholds.json"))

    # Evaluate test.
    print("Evaluating test split ...")
    all_rows = []
    for i, sample in enumerate(test_samples):
        print(f"  sample {i+1}/{len(test_samples)} | {preview_text(sample['question'], 90)}")
        rows = run_sample(sample)
        all_rows.extend(rows)
        free_memory()

    df = pd.DataFrame(all_rows)
    save_df(df, os.path.join(CSV_DIR, "raw_variant_rows.csv"))

    # Apply thresholds.
    df = apply_thresholds(df, thresholds)
    save_df(df, os.path.join(CSV_DIR, "all_results.csv"))

    # Question-level and variant-level summaries.
    variant_summary = summarize_variant(df)
    question_summary = summarize_question_level(df)
    question_variant_grid = summarize_question_variant_grid(df)

    save_df(variant_summary, os.path.join(CSV_DIR, "variant_summary.csv"))
    save_df(question_summary, os.path.join(CSV_DIR, "question_summary.csv"))
    save_df(question_variant_grid, os.path.join(CSV_DIR, "question_variant_grid.csv"))

    # Outcome tables.
    transition_counts = df.groupby(["variant", "transition"], as_index=False).size().rename(columns={"size": "count"})
    save_df(transition_counts, os.path.join(CSV_DIR, "transition_counts.csv"))

    # Basin metrics.
    basin_heatmaps_info = []
    for variant in df["variant"].unique().tolist():
        sdf = df[df["variant"] == variant].copy()
        if len(sdf) == 0:
            continue
        basin_heatmaps_info.append({
            "variant": variant,
            "stability_score": float((1.0 - sdf["pass1_vs_pass2_prediction_changed"].mean())),
            "transition_entropy": float(entropy_from_probs(np.array(list(Counter(sdf["transition"]).values()), dtype=np.float64) / max(1, len(sdf)))),
        })
    basin_summary = pd.DataFrame(basin_heatmaps_info)
    save_df(basin_summary, os.path.join(CSV_DIR, "basin_summary.csv"))

    # Global summary.
    overall = {
        "n_questions": int(question_summary["question_id"].nunique()),
        "n_variants": int(df["variant"].nunique()),
        "baseline_variant": BASELINE_VARIANT,
        "baseline_pass1_raw_accuracy": float(variant_summary[variant_summary["variant"] == BASELINE_VARIANT]["pass1_raw_accuracy"].iloc[0]),
        "baseline_pass2_raw_accuracy": float(variant_summary[variant_summary["variant"] == BASELINE_VARIANT]["pass2_raw_accuracy"].iloc[0]),
        "baseline_pass1_cal_accuracy": float(variant_summary[variant_summary["variant"] == BASELINE_VARIANT]["pass1_cal_accuracy"].iloc[0]),
        "baseline_pass2_cal_accuracy": float(variant_summary[variant_summary["variant"] == BASELINE_VARIANT]["pass2_cal_accuracy"].iloc[0]),
        "baseline_improvement_rate": float(variant_summary[variant_summary["variant"] == BASELINE_VARIANT]["improvement_rate"].iloc[0]),
        "baseline_regression_rate": float(variant_summary[variant_summary["variant"] == BASELINE_VARIANT]["regression_rate"].iloc[0]),
        "baseline_transition_entropy": float(variant_summary[variant_summary["variant"] == BASELINE_VARIANT]["transition_entropy"].iloc[0]),
        "baseline_q_stability_mean": float(variant_summary[variant_summary["variant"] == BASELINE_VARIANT]["q_stability_mean"].iloc[0]),
        "baseline_q_drift_mean": float(variant_summary[variant_summary["variant"] == BASELINE_VARIANT]["q_drift_mean"].iloc[0]),
    }
    save_json(overall, os.path.join(REPORTS_DIR, "overall_summary.json"))

    # --- NEW VISUALIZATIONS ---
    grouped_bar_plot(
        variant_summary["variant"],
        variant_summary["pass1_raw_accuracy"],
        variant_summary["pass2_raw_accuracy"],
        "Pass-1", "Pass-2",
        "Pass-1 vs Pass-2 Accuracy Comparison",
        os.path.join(PLOTS_DIR, "accuracy_comparison_grouped.png")
    )

    violin_plot_distributions(df, "pass1_logodds", "variant", "Pass-1 Log-odds Distribution by Variant", os.path.join(PLOTS_DIR, "violin_pass1_logodds.png"), "Log-odds")
    violin_plot_distributions(df, "pass2_logodds", "variant", "Pass-2 Log-odds Distribution by Variant", os.path.join(PLOTS_DIR, "violin_pass2_logodds.png"), "Log-odds")
    violin_plot_distributions(df, "delta_logodds", "variant", "Delta Log-odds (Pass 2 - Pass 1) by Variant", os.path.join(PLOTS_DIR, "violin_delta_logodds.png"), "Δ Log-odds")
    # --------------------------

    # Per-variant dashboards.
    plot_variant_metric_dashboard(variant_summary, os.path.join(PLOTS_DIR, "variant_metrics_dashboard.png"))
    plot_waterfall(variant_summary, os.path.join(PLOTS_DIR, "self_correction_waterfall.png"))
    plot_phase_portrait(df[df["variant"] == BASELINE_VARIANT], os.path.join(PLOTS_DIR, "phase_portrait_baseline.png"))
    plot_transition_heatmap(df[df["variant"] == BASELINE_VARIANT], os.path.join(PLOTS_DIR, "transition_heatmap_baseline.png"))
    quiver_transition_plot(df[df["variant"] == BASELINE_VARIANT], os.path.join(PLOTS_DIR, "trajectory_field_baseline.png"))
    strip_transition_plot(df[df["variant"] == BASELINE_VARIANT], os.path.join(PLOTS_DIR, "strip_transitions_baseline.png"))
    plot_response_drift(df[df["variant"] == BASELINE_VARIANT], os.path.join(PLOTS_DIR, "response_drift_baseline.png"))
    plot_bifurcation(df[df["variant"] == BASELINE_VARIANT], os.path.join(PLOTS_DIR, "bifurcation_baseline.png"))
    plot_basin_map(df[df["variant"] == BASELINE_VARIANT], os.path.join(PLOTS_DIR, "basin_atlas_baseline.png"))
    plot_basin_heatmaps(df[df["variant"] == BASELINE_VARIANT], "baseline")

    # Reliability diagrams for baseline (both passes) and best pass2 style.
    best_style = variant_summary.sort_values("pass2_raw_accuracy", ascending=False).iloc[0]["variant"]
    rel_tables = []
    for variant in [BASELINE_VARIANT, best_style]:
        sdf = df[df["variant"] == variant]
        r1 = plot_reliability(sdf, variant, "pass1", os.path.join(PLOTS_DIR, f"reliability_{variant}_pass1.png"))
        r2 = plot_reliability(sdf, variant, "pass2", os.path.join(PLOTS_DIR, f"reliability_{variant}_pass2.png"))
        r1["variant"] = variant
        r1["pass"] = "pass1"
        r2["variant"] = variant
        r2["pass"] = "pass2"
        rel_tables.extend([r1, r2])
    if rel_tables:
        rel_df = pd.concat(rel_tables, ignore_index=True)
        save_df(rel_df, os.path.join(CSV_DIR, "reliability_tables.csv"))

    # Drift / strip / trajectory plots for all variants as a global atlas.
    plot_mean_trajectory(df, os.path.join(PLOTS_DIR, "trajectory_field_all_variants.png"))
    plot_strip_transitions(df, os.path.join(PLOTS_DIR, "strip_transitions_all_variants.png"))
    plot_response_drift(df, os.path.join(PLOTS_DIR, "response_drift_all_variants.png"))
    plot_bifurcation(df, os.path.join(PLOTS_DIR, "bifurcation_all_variants.png"))
    plot_basin_map(df, os.path.join(PLOTS_DIR, "basin_atlas_all_variants.png"))
    plot_basin_heatmaps(df, "all_variants")

    # Transition Sankey.
    if PLOTLY_AVAILABLE:
        for variant in [BASELINE_VARIANT, best_style]:
            sdf = df[df["variant"] == variant].copy()
            source = []
            target = []
            value = []
            nodes = ["W→W", "W→C", "C→W", "C→C"]
            idx = {n: i for i, n in enumerate(nodes)}
            links = Counter(sdf["transition"].tolist())
            # stage 1 -> stage 2 (transitions to transition states)
            for s in ["wrong→wrong", "wrong→correct", "correct→wrong", "correct→correct"]:
                source.append(0)
                target.append(idx[s])
                value.append(int(links.get(s, 0)))
            fig = go.Figure(go.Sankey(
                arrangement="snap",
                node=dict(pad=18, thickness=20, label=["Start"] + nodes, color=["rgba(31,119,180,0.7)"] + ["rgba(100,100,100,0.65)"] * 4),
                link=dict(source=source, target=[t + 1 for t in target], value=value, color=["rgba(120,120,120,0.25)"] * len(value)),
            ))
            fig.update_layout(title_text=f"Self-correction transition Sankey ({variant})", font=dict(size=12), width=1200, height=700)
            try:
                fig.write_html(os.path.join(HTML_DIR, f"sankey_{variant}.html"))
            except Exception:
                pass
            try:
                fig.write_image(os.path.join(PLOTS_DIR, f"sankey_{variant}.png"), scale=2)
            except Exception:
                pass

    # Representative question dashboards.
    plot_representative_trajectories(df[df["variant"] == BASELINE_VARIANT], os.path.join(PLOTS_DIR, "representative_trajectories_baseline.png"))
    plot_question_glyph_atlas(df[df["variant"] == BASELINE_VARIANT], os.path.join(PLOTS_DIR, "question_glyph_atlas_baseline.png"), top_n=12)

    # Global question summary plots.
    q = question_summary.copy().sort_values(["stability_score", "transition_entropy"], ascending=[True, False])
    bar_plot(q["question_id"].astype(str).tolist(), q["stability_score"].tolist(), "Question stability scores", os.path.join(PLOTS_DIR, "question_stability_scores.png"), ylabel="Stability", rotate=90)
    histogram(q["transition_entropy"].tolist(), "Question transition entropy distribution", os.path.join(PLOTS_DIR, "question_transition_entropy_hist.png"), xlabel="Entropy")
    histogram(q["response_drift_mean"].tolist(), "Question response drift distribution", os.path.join(PLOTS_DIR, "question_response_drift_hist.png"), xlabel="Drift")
    scatter(q["stability_score"].tolist(), q["pass2_accuracy"].tolist(), "Stability vs pass-2 accuracy", os.path.join(PLOTS_DIR, "stability_vs_pass2_accuracy.png"), xlabel="Stability", ylabel="Pass-2 accuracy")
    scatter(q["transition_entropy"].tolist(), q["improvement_rate"].tolist(), "Transition entropy vs improvement rate", os.path.join(PLOTS_DIR, "entropy_vs_improvement.png"), xlabel="Transition entropy", ylabel="Improvement rate")
    scatter(q["response_drift_mean"].tolist(), q["delta_logodds_mean"].tolist(), "Response drift vs Δ log-odds", os.path.join(PLOTS_DIR, "drift_vs_delta_logodds.png"), xlabel="Response drift", ylabel="Δ log-odds")

    # Question-level basin classes.
    basin_counts = q["basin_class"].value_counts().reset_index()
    basin_counts.columns = ["basin_class", "count"]
    save_df(basin_counts, os.path.join(CSV_DIR, "basin_class_counts.csv"))
    bar_plot(basin_counts["basin_class"].tolist(), basin_counts["count"].tolist(), "Question basin class counts", os.path.join(PLOTS_DIR, "basin_class_counts.png"), ylabel="Count", rotate=20)

    # Basin / grid heatmaps.
    basin_grid_outputs = []
    for variant in [BASELINE_VARIANT, best_style]:
        sdf = df[df["variant"] == variant].copy()
        basin_grid_outputs.append((variant, plot_delta_heatmap(sdf, "pass1_logodds", "pass1_commitment", "improved", f"Improvement basin ({variant})", os.path.join(PLOTS_DIR, f"improvement_basin_{variant}.png"))))
        basin_grid_outputs.append((variant, plot_delta_heatmap(sdf, "pass1_logodds", "pass1_commitment", "regressed", f"Regression basin ({variant})", os.path.join(PLOTS_DIR, f"regression_basin_{variant}.png"))))
        basin_grid_outputs.append((variant, plot_delta_heatmap(sdf, "pass1_logodds", "pass1_commitment", "pass2_correct", f"Pass-2 correctness basin ({variant})", os.path.join(PLOTS_DIR, f"pass2_correct_basin_{variant}.png"))))
        basin_grid_outputs.append((variant, plot_delta_heatmap(sdf, "pass1_logodds", "pass1_commitment", "transition_entropy", f"Transition entropy basin ({variant})", os.path.join(PLOTS_DIR, f"transition_entropy_basin_{variant}.png"))))

    # Global dashboard.
    fig, axes = plt.subplots(2, 2, figsize=(18, 11), constrained_layout=True)
    axes[0, 0].bar(variant_summary["variant"], variant_summary["pass1_raw_accuracy"], alpha=0.85)
    axes[0, 0].set_title("Pass-1 raw accuracy by variant")
    axes[0, 0].tick_params(axis="x", rotation=30)
    annotated_bar(axes[0, 0])

    axes[0, 1].bar(variant_summary["variant"], variant_summary["pass2_raw_accuracy"], alpha=0.85)
    axes[0, 1].set_title("Pass-2 raw accuracy by variant")
    axes[0, 1].tick_params(axis="x", rotation=30)
    annotated_bar(axes[0, 1])

    axes[1, 0].bar(variant_summary["variant"], variant_summary["improvement_rate"], alpha=0.85)
    axes[1, 0].set_title("Improvement rate by variant")
    axes[1, 0].tick_params(axis="x", rotation=30)
    annotated_bar(axes[1, 0])

    axes[1, 1].bar(variant_summary["variant"], variant_summary["regression_rate"], alpha=0.85)
    axes[1, 1].set_title("Regression rate by variant")
    axes[1, 1].tick_params(axis="x", rotation=30)
    annotated_bar(axes[1, 1])

    plt.suptitle("Self-correction basin atlas dashboard", fontsize=16, y=1.01)
    plt.savefig(os.path.join(PLOTS_DIR, "overall_dashboard.png"), dpi=180, bbox_inches="tight")
    plt.close()

    # Variant metric heatmap.
    variant_metric = variant_summary.set_index("variant")[[
        "pass1_raw_accuracy", "pass2_raw_accuracy", "pass1_cal_accuracy", "pass2_cal_accuracy",
        "pass1_commitment_mean", "pass2_commitment_mean", "improvement_rate", "regression_rate",
        "transition_entropy", "q_stability_mean", "q_drift_mean"
    ]]
    heatmap(variant_metric.values, variant_metric.columns.tolist(), variant_metric.index.tolist(), "Variant metric heatmap", os.path.join(PLOTS_DIR, "variant_metric_heatmap.png"), xlabel="Metric", ylabel="Variant", cmap="coolwarm")

    # Save a combined response atlas CSVs for later analysis.
    save_df(df, os.path.join(CSV_DIR, "all_results_with_calibration.csv"))
    save_df(question_summary, os.path.join(CSV_DIR, "question_summary.csv"))
    save_df(variant_summary, os.path.join(CSV_DIR, "variant_summary.csv"))

    # Markdown report.
    md = ["# StrategyQA self-correction basin atlas report\n"]
    md.append(f"- Model: `{BASE_MODEL}`")
    md.append(f"- Validation samples: {N_VALIDATION}")
    md.append(f"- Test samples: {N_TEST}")
    md.append(f"- Prompt variants: {', '.join([v.name for v in VARIANTS])}")
    md.append(f"- Baseline variant: `{BASELINE_VARIANT}`")
    md.append(f"- Deterministic decoding: {TEMPERATURE == 0.0}")
    md.append("\n## Overall summary\n")
    md.append("| Metric | Value |\n|---|---:|")
    for k, v in overall.items():
        if isinstance(v, (int, float)):
            md.append(f"| {k} | {v:.3f} |")
        else:
            md.append(f"| {k} | {v} |")

    md.append("\n## Variant summary\n")
    md.append("| Variant | Pass1 acc | Pass2 acc | Pass1 cal acc | Pass2 cal acc | Improve | Regress | Transition entropy | Drift |")
    md.append("|---|---:|---:|---:|---:|---:|---:|---:|---:|")
    for _, r in variant_summary.sort_values("strength").iterrows():
        md.append(
            f"| {r['variant']} | {r['pass1_raw_accuracy']:.3f} | {r['pass2_raw_accuracy']:.3f} | {r['pass1_cal_accuracy']:.3f} | {r['pass2_cal_accuracy']:.3f} | {r['improvement_rate']:.3f} | {r['regression_rate']:.3f} | {r['transition_entropy']:.3f} | {r['mean_response_drift']:.3f} |"
        )

    md.append("\n## Question basin classes\n")
    md.append("| Basin class | Count |\n|---|---:|")
    for _, r in basin_counts.iterrows():
        md.append(f"| {r['basin_class']} | {int(r['count'])} |")

    md.append("\n## Interpretation\n")
    md.append("- A high stability score means the question rarely flips across prompt variants and passes.")
    md.append("- A high transition entropy means the sample moves through many different states (e.g. wrong→correct, correct→wrong).")
    md.append("- The phase portraits show whether pass2 sharpens or destabilizes the decision basin.")
    md.append("- The quiver map shows the direction of self-correction in confidence-commitment space.")
    md.append("- The improvement and regression basins reveal where self-correction is genuinely helpful and where it over-corrects.")
    md.append("- The response-drift plots show whether explanation drift aligns with confidence changes.")
    md.append("- The glyph atlas makes individual question behavior visually inspectable at a glance.")

    with open(os.path.join(REPORTS_DIR, "report.md"), "w") as f:
        f.write("\n".join(md))

    save_json({
        "config": {
            "BASE_MODEL": BASE_MODEL,
            "SFT_PATH": SFT_PATH,
            "N_VALIDATION": N_VALIDATION,
            "N_TEST": N_TEST,
            "VARIANTS": [asdict(v) for v in VARIANTS],
            "BASELINE_VARIANT": BASELINE_VARIANT,
            "PASS1_MAX_NEW": PASS1_MAX_NEW,
            "PASS2_MAX_NEW": PASS2_MAX_NEW,
            "TEMPERATURE": TEMPERATURE,
            "TOP_P": TOP_P,
        },
        "overall": overall,
        "variant_summary": variant_summary.to_dict(orient="records"),
        "question_summary": question_summary.to_dict(orient="records"),
    }, os.path.join(REPORTS_DIR, "summary.json"))

    print("\n===== OVERALL =====")
    print(json.dumps(overall, indent=2))
    print("\n===== VARIANT SUMMARY =====")
    print(variant_summary.to_string(index=False))
    print("\n===== QUESTION SUMMARY (top rows) =====")
    print(question_summary.head(10).to_string(index=False))
    print("\nSaved outputs to:")
    print(f"- {OUT_DIR}")
    print(f"- CSVs: {CSV_DIR}")
    print(f"- Plots: {PLOTS_DIR}")
    print(f"- HTML: {HTML_DIR}")
    print(f"- Reports: {REPORTS_DIR}")
    print(f"- NPZ: {NPZ_DIR}")

    free_memory(MODEL)

if __name__ == "__main__":
    main()

Loading model and tokenizer ...


config.json:   0%|          | 0.00/967 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

...

data/test-00000-of-00001-bae602f3ee37f4c(…):   0%|          | 0.00/161k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1603 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/687 [00:00<?, ? examples/s]

Calibrating thresholds on validation split ...
Evaluating test split ...
  sample 1/120 | Could boolean algebra be described as binary?
  sample 2/120 | Would lumberjacks get full after eating three dosa?
  sample 3/120 | Would a member of the United States Air Force get a discount at Dunkin Donuts?
  sample 4/120 | In a hypothetical race between a Swallow and an American Woodcock, would the Swallow win?
  sample 5/120 | Were number of states in Ancient Greece underwhelming compared to US states in 1900?
  sample 6/120 | Can petroleum jelly be used as fuel in a car?
  sample 7/120 | Did Julia Roberts practice blast beats as a child?
  sample 8/120 | Is myocardial infarction a brain problem?
  sample 9/120 | Would a hedgehog avoid animals without a spinal cord?
  sample 10/120 | Can The Hobbit be read in its entirety in four minutes?
  sample 11/120 | Could Plato have agreed with the beliefs of Jainism?
  sample 12/120 | Can Michael Bloomberg fund the debt of Micronesia for a decade?
  

NameError: name 'plot_waterfall' is not defined